# 2. Google News Local pages, Ontario municipalities

Reads each town's Google News Local page (logged out, English and French) and saves its story cards. A page can't be read for a past date, so each run captures that day only.

- **Input:** `data/raw/local_page_urls.csv`, one Local page address per town. Each address is built from the town's Google Knowledge Graph ID (from Wikidata, checked against its StatCan code).
- **Output:** `outputs/local_page_cards_<run>.csv`.

At 30 s between requests, the 40-town sample (`SAMPLE_ONLY = True`, 80 pages) takes about 45 minutes; all 390 towns (780 pages) take about 7 hours. The run stops at a block or a CAPTCHA.

In [ ]:
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'collection' else Path.cwd()   # repository root
RAW, CACHE, OUT = ROOT / 'data' / 'raw', ROOT / 'cache', ROOT / 'outputs'
CACHE.mkdir(exist_ok=True); OUT.mkdir(exist_ok=True)
import datetime as dt, gzip, html, random, re, time
import pandas as pd, requests

SAMPLE_ONLY = True     # True = the 40-town sample; False = every town
GAP = 30               # seconds between requests
EDITIONS = {'en-CA': 'hl=en-CA&gl=CA&ceid=CA:en', 'fr-CA': 'hl=fr-CA&gl=CA&ceid=CA:fr'}
STOP = ('consent.google.com', '/sorry/', 'unusual traffic', 'captcha')
HEADERS = {'User-Agent': 'INSE6450 student research (Google News Local pages)'}

towns = pd.read_csv(RAW / 'local_page_urls.csv', dtype={'town_id': str})
if SAMPLE_ONLY:
    towns = towns[towns.sample_40]
print(f"{len(towns)} towns x {len(EDITIONS)} editions = {len(towns) * len(EDITIONS)} pages")

## Step 1. Read the pages

In [ ]:
run = dt.datetime.now(dt.timezone.utc).strftime('%Y%m%dT%H%MZ')
run_dir = CACHE / 'local_pages' / run
run_dir.mkdir(parents=True, exist_ok=True)
for i, (t, ed) in enumerate((t, ed) for t in towns.itertuples() for ed in EDITIONS):
    if i:
        time.sleep(GAP + random.uniform(0, GAP / 4))
    r = requests.get(f'{t.topic_url}?{EDITIONS[ed]}', headers={**HEADERS, 'Accept-Language': ed}, timeout=60)
    print(f'{i + 1}. {t.short_name} {ed}: {r.status_code}')
    if r.status_code in (403, 429, 503) or any(s in r.url or s in r.text[:20000].lower() for s in STOP):
        print('Stopped: Google answered with a block or a CAPTCHA. Try again another day.')
        break
    if r.status_code == 200:
        with gzip.open(run_dir / f'{t.town_id}_{ed}.html.gz', 'wt', encoding='utf-8') as f:
            f.write(r.text)

## Step 2. Read the story cards

Each card on the page has:
- a hidden link (`class="WwrzSb"`) whose `jslog` attribute carries the original article URL in base64;
- the outlet's name in `class="vr1PYe"`;
- the time in `<time datetime>`;
- the headline in a `./read/...` link.

Cards are numbered in page order (`rank`).

In [ ]:
import base64

def b64text(s):
    try:
        return base64.b64decode(s + '=' * (-len(s) % 4)).decode('utf-8', 'replace')
    except ValueError:
        return ''

def parse_cards(page):
    starts = [m.start() for m in re.finditer(r'<a class="WwrzSb"', page)]
    cards, seen = [], set()
    for k, st in enumerate(starts):
        chunk = page[st: starts[k + 1] if k + 1 < len(starts) else st + 20000]
        link = re.search(r'jslog="95014; 5:([A-Za-z0-9+/=]+)', chunk)
        url = re.search(r'https?://[^"\\]+', b64text(link.group(1))) if link else None
        source = re.search(r'class="vr1PYe"[^>]*>([^<]+)<', chunk)
        when = re.search(r'datetime="([^"]+)"', chunk)
        heads = [html.unescape(re.sub(r'<[^>]+>', '', h)).strip()
                 for h in re.findall(r'<a[^>]*href="\./read/[^"]*"[^>]*>(.*?)</a>', chunk, flags=re.S)]
        head = max([h for h in heads if h], key=len, default='')
        if head and head not in seen:
            seen.add(head)
            cards.append({'rank': len(cards) + 1, 'headline': head, 'source': html.unescape(source.group(1)) if source else '',
                          'time': when.group(1) if when else '', 'link': url.group(0) if url else ''})
    return cards

rows = []
for p in sorted(run_dir.glob('*.html.gz')):
    town_id, ed = p.name.split('.')[0].split('_')
    page = gzip.open(p, 'rt', encoding='utf-8', errors='replace').read()
    rows += [{'run': run, 'town_id': town_id, 'edition': ed, **c} for c in parse_cards(page)]
cards = pd.DataFrame(rows)
cards.to_csv(OUT / f'local_page_cards_{run}.csv', index=False)
print(f'{cards.town_id.nunique()} towns, {len(cards)} cards')
cards.head()

## Step 3. Quick look: how much of each page is about the town?

A rough check: the share of cards whose headline names the town. The tags in `data/google_news_local_pages/` are stricter: they also count the town's communities and its own outlets, and every race story was checked by hand.

In [ ]:
c = cards.merge(towns[['town_id', 'short_name', 'population_2021']], on='town_id')
c['names_town'] = [n.lower() in h.lower() for n, h in zip(c.short_name, c.headline)]
by_town = c[c.edition == 'en-CA'].groupby(['short_name', 'population_2021']).names_town.mean().reset_index()
by_town['size'] = pd.cut(by_town.population_2021, [0, 2000, 10000, 50000, 1e8], labels=['<2k', '2k-10k', '10k-50k', '50k+'])
by_town.groupby('size', observed=True).names_town.agg(['mean', 'count']).rename(columns={'mean': 'share of cards naming the town', 'count': 'towns'})